## **Race Data Cleaning**

##### **Imports**

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

#### **Importing FastF1 Lap, Weather, and Data (2018-2025)**

**Lap Data:** Contains detailed lap-by-lap performance data for every driver across all races, including lap times, sector times, position, team, tyre info, and track conditions.

**Weather Data:** Contains minute-level weather measurements for each race, including temperature, humidity, wind, and rainfall conditions. 

**Event Data:** Contains race data, including event names, race schedules, start times, and circuit location information.

In [ ]:
# Import and combine all f1 lap data
laps_directory = Path('../data/ff1_laps_data')
laps_files = laps_directory.glob('laps_data_*.csv')

laps_raw = pd.concat(
    [pd.read_csv(file) for file in laps_files],
    ignore_index=True
)

# laps_raw.head()

In [ ]:
# Import and combine all f1 weather data
weather_directory = Path('../data/ff1_weather_data')
weather_files = weather_directory.glob('weather_data_*.csv')

weather_data = pd.concat(
    [pd.read_csv(file) for file in weather_files],
    ignore_index=True
)

# weather_data.head()

In [ ]:
# Import event location data
event_data = pd.read_csv('../data/f1_event_location_data.csv')

# event_data.head()

## **Part 1 - Lap Data Cleaning**

### **Standardizing Data Types**

We standardize key data types before analysis to ensure the dataset can be used consistently. 

- Timing columns are converted to timedelta values for accurate time calculations.
- Boolean fields are converted to true boolean types. 
- Entirely blank columns are removed.

In [ ]:
# Standardize time data types to timedelta
time_columns = [column for column in laps_raw.keys() if 'Time' in column]
laps_raw[time_columns] = laps_raw[time_columns].apply(pd.to_timedelta, errors='coerce')

In [ ]:
# Identify boolean columns
bool_columns = ["IsPersonalBest", "FastF1Generated", "IsAccurate"]
laps_raw[["IsPersonalBest", "FastF1Generated", "IsAccurate"]].dtypes

In [ ]:
# Update non-boolean values
laps_raw["IsPersonalBest"] = laps_raw["IsPersonalBest"].astype(bool)

In [ ]:
# Drop columns with all null values
laps_raw = laps_raw.drop(['Deleted', 'DeletedReason', 'LapStartDate'], axis=1)

### **Handling Missing Data**

A major source of missing data occurs in the **`LapTime`** column. In the FastF1 dataset, missing lap times appear to be commonly associated with pit laps, retirements, or incomplete laps. However, both 'LapStartTime' and 'Time' have no null values, meaning 'LapTime' can be reconstructed by subtracting the lap start time from the lap end time for every missing row. 

Most remaining missing values occur on the first lap for each driver. These missing fields commonly include Stint, sector times, and sector session times.

1) For Stint, missing values on lap one are filled with 1.
2) For sector times, missing values are first reconstructed when enough timing information is available. 
3) If two sector times are missing, the remaining lap time is split evenly between the missing sectors.
4) If all three sector times are missing, the full LapTime is divided evenly across Sector1Time, Sector2Time, and Sector3Time.
5) After sector times are filled, sector session times are reconstructed cumulatively using LapStartTime and the completed sector time values.

**Notes:**
- Terminal Laps for drivers that did not finish the race and are missing position for the given lap are given the last possible position.
- Track status is assumed to be clear if no other flag is shown.

**`LapTime`**

In [ ]:
# Reconstruct missing LapTime
# LapTime = "Time" − "LapStartTime"
mask = laps_raw["LapTime"].isna()
reconstructed_lap_time = laps_raw["Time"] - laps_raw["LapStartTime"]

laps_raw.loc[mask, "LapTime"] = reconstructed_lap_time[mask]

**`Stint`**

In [ ]:
# Set missing stint on first lap to 1
laps_raw.loc[(laps_raw["LapNumber"] == 1) & laps_raw["Stint"].isna(), "Stint"] = 1

# Forward fill remaining null values
laps_raw['Stint'] = (
    laps_raw.groupby(['Year', 'EventName', 'Driver'])['Stint'].ffill()
)

**`SectorTimes`  `SectorSessionTimes`**

In [ ]:
# Sector1Time = LapTime - Sector2Time - Sector3Time
mask = (
    laps_raw["Sector1Time"].isna()
    & laps_raw["Sector2Time"].notna()
    & laps_raw["Sector3Time"].notna()
)

laps_raw.loc[mask, "Sector1Time"] = (
    laps_raw.loc[mask, "LapTime"]
    - laps_raw.loc[mask, "Sector2Time"]
    - laps_raw.loc[mask, "Sector3Time"]
)

# Sector2Time = LapTime - Sector1Time - Sector3Time
mask = (
    laps_raw["Sector2Time"].isna()
    & laps_raw["Sector1Time"].notna()
    & laps_raw["Sector3Time"].notna()
)

laps_raw.loc[mask, "Sector2Time"] = (
    laps_raw.loc[mask, "LapTime"]
    - laps_raw.loc[mask, "Sector1Time"]
    - laps_raw.loc[mask, "Sector3Time"]
)

# Sector3Time = LapTime - Sector1Time - Sector2Time
mask = (
    laps_raw["Sector3Time"].isna()
    & laps_raw["Sector1Time"].notna()
    & laps_raw["Sector2Time"].notna()
)

laps_raw.loc[mask, "Sector3Time"] = (
    laps_raw.loc[mask, "LapTime"]
    - laps_raw.loc[mask, "Sector1Time"]
    - laps_raw.loc[mask, "Sector2Time"]
)

In [ ]:
# Estimate missing sector times
def fill_sector_times(row):
    sectors = ["Sector1Time", "Sector2Time", "Sector3Time"]

    missing = [s for s in sectors if pd.isna(row[s])]
    present = [s for s in sectors if pd.notna(row[s])]

    # All 3 missing
    if len(missing) == 3:
        value = row["LapTime"] / 3

        for s in sectors:
            row[s] = value

    # Exactly 2 missing
    elif len(missing) == 2 and len(present) == 1:
        remaining = row["LapTime"] - row[present[0]]
        value = remaining / 2

        for s in missing:
            row[s] = value

    return row

laps_raw = laps_raw.apply(fill_sector_times, axis=1)

In [ ]:
# Recalculate sector session times
mask = laps_raw["Sector1SessionTime"].isna()
laps_raw.loc[mask, "Sector1SessionTime"] = (
    laps_raw.loc[mask, "LapStartTime"] + laps_raw.loc[mask, "Sector1Time"]
)

mask = laps_raw["Sector2SessionTime"].isna()
laps_raw.loc[mask, "Sector2SessionTime"] = (
    laps_raw.loc[mask, "Sector1SessionTime"] + laps_raw.loc[mask, "Sector2Time"]
)

mask = laps_raw["Sector3SessionTime"].isna()
laps_raw.loc[mask, "Sector3SessionTime"] = (
    laps_raw.loc[mask, "Sector2SessionTime"] + laps_raw.loc[mask, "Sector3Time"]
)

**`Position`**

In [ ]:
# Fill position of terminal laps (DNF)
last_position = (
    laps_raw.groupby(['Year', 'EventName', 'LapNumber'])['Position']
    .transform('max') + 1
)

laps_raw['Position'] = laps_raw['Position'].fillna(last_position)

**`TrackStatus`**

In [ ]:
# Assume track status == 1 if no status given
laps_raw.loc[laps_raw['TrackStatus'].isna(), 'TrackStatus'] = 1

**`TyreLife`**

Missing values are reconstructed using neighboring laps within each race and driver grouping

- Function fills missing values forward and backward based on surrounding tire life progression
- Any fully missing groups defaulting to 0.

In [ ]:
# Use previous and next rows to fill in tyre data
def calculate_tyre_life(laps):
    laps = laps.copy()
    
    # Handle trailing NaNs
    for i in range(1, len(laps)):
        if pd.isna(laps.iloc[i]) and pd.notna(laps.iloc[i-1]):
            laps.iloc[i] = laps.iloc[i-1] + 1
            
    laps = laps[::-1] # reverse lap data
    
    # Handle leading NaNs
    for i in range(1, len(laps)):
        if pd.isna(laps.iloc[i]):
            laps.iloc[i] = laps.iloc[i-1] - 1
            
    return laps[::-1] # return original order
        
# Fill in missing tyre data
laps_raw['TyreLife'] = (
    laps_raw.groupby(['Year', 'Location', 'EventName', 'Driver'])['TyreLife'].transform(calculate_tyre_life)
)

# Does not catch all NaN groups
laps_raw['TyreLife'] = laps_raw['TyreLife'].fillna(0)

 **`Compound`**
- Missing values are converted from the string 'nan' to true null values
- Backward fill used to infer the compound from surrounding laps within the same race

In [ ]:
# Replace Compound 'nan' string with np.nan
laps_raw['Compound'] = laps_raw['Compound'].replace('nan', np.nan)

# Backward fill missing tire compound values
laps_raw['Compound'] = (
    laps_raw.groupby(['Year', 'Location', 'EventName', 'Driver'])['Compound'].bfill()
)

 **`Speeds`**

In [ ]:
speed_cols = ['SpeedFL', 'SpeedI1', 'SpeedI2', 'SpeedST']

laps_raw[speed_cols] = (
    laps_raw.groupby(['Year', 'EventName', 'Driver'])[speed_cols]
            .transform(lambda x: x.interpolate())
)

### **Standardizing Identities** 

#### **Location Names**

In the FastF1 dataset, certain **`Location`** values may appear under different names despite referring to the same circuit or race location. To improve consistency across the dataset, we use the replace() method to standardize these names into a single naming convention.

In [ ]:
# Normalize location names 
laps_raw['Location'] = laps_raw['Location'].replace({
    'Monte Carlo': 'Monaco',
    'Marina Bay': 'Singapore',
    'Miami Gardens': 'Miami',
    'Yas Island': 'Yas Marina'
})

#### **Team Lineage**

Teams in Formula 1 are frequently renamed over time due to changes in sponsorship, ownership, or manufacturer branding. To maintain consistency across seasons, we will create a dictionary that maps historical team names to their standardized 2025 team names. This normalization process will support more accurate multi-season analysis and allow us to better evaluate team performance over time.

Website for tracking team changes over time: _[Formula 1 Lineage](https://flamingtempura.github.io/formula1-lineage/)_

**Author:** Peter West (FlamingTempura)

In [ ]:
team_lineage = {
    # Racing Bulls lineage
    "Toro Rosso": "Racing Bulls",
    "AlphaTauri": "Racing Bulls",
    "RB": "Racing Bulls",
    "Racing Bulls": "Racing Bulls",

    # Kick Sauber lineage
    "Sauber": "Kick Sauber",
    "Alfa Romeo Racing": "Kick Sauber",
    "Alfa Romeo": "Kick Sauber",
    "Kick Sauber": "Kick Sauber",

    # Aston Martin lineage
    "Force India": "Aston Martin",
    "Racing Point": "Aston Martin",
    "Aston Martin": "Aston Martin",

    # Alpine lineage
    "Renault": "Alpine",
    "Alpine": "Alpine",

    # Stable teams
    "McLaren": "McLaren",
    "Williams": "Williams",
    "Haas F1 Team": "Haas",
    "Red Bull Racing": "Red Bull Racing",
    "Mercedes": "Mercedes",
    "Ferrari": "Ferrari",
}

# Standardize names for teams across seasons
laps_raw['Team'] = laps_raw['Team'].apply(lambda x: team_lineage[x])

#### **Lap Times**

We merge in **`StartTime`** to calculate each lap’s exact start time in UTC. This allows us to align lap data with weather data and capture the conditions at the moment each lap occurred.

In [ ]:
# Convert start time type
event_data['StartTime'] = pd.to_datetime(event_data['StartTime'])

# Add event name to handle repeat locations
laps_raw = laps_raw.merge(
    event_data[['Year', 'EventName', 'Location', 'StartTime']],
    on=['Year', 'EventName', 'Location'],
    how='left'
)

# Add a UTC lap start time for merging with weather data
laps_raw['LapStartTimeUTC'] = laps_raw['LapStartTime'] + laps_raw['StartTime']

laps_raw = laps_raw.drop(['StartTime'], axis=1)

# laps_raw.sample(5)

#### **Feature Engineering**

We create boolean flags to identify pit laps and terminal laps.

- **`IsPitLap`** marks laps with pit-in or pit-out times
- **`IsTerminalLap`** identifies each driver’s final recorded lap in a race

In [ ]:
# Add pit lap feature
laps_raw["IsPitLap"] = (laps_raw["PitInTime"].notna() | laps_raw["PitOutTime"].notna())

# Add terminal lap feature
laps_raw["IsTerminalLap"] = (
    laps_raw["LapNumber"] == laps_raw.groupby(["Year", "Location", "EventName", "Driver"])["LapNumber"].transform("max")
)

## **Part 2 - Combining Lap & Weather Data**

### **Overview**

To evaluate the impact of weather on driver performance, the lap and weather datasets must be combined into a single file. However, the two datasets are recorded at different levels of granularity. Weather data was recorded at even one-minute intervals, while lap start times depend on a driver's pace. To address this issue, the weather data was transformed to provide weather estimates at one-second intervals, allowing each lap to be matched with the closest available weather observation. This should create a more consistent timeframe across both datasets and enable more accurate analysis of how changing weather conditions influence lap and sector performance throughout a race.

**Standardizing Data Types & Location Names**

We standardize key data types before analysis to ensure the dataset can be used consistently. 

- Timing columns for both weather and event data are converted to timedelta values for accurate time calculations.
- **`Location`** is updated with the same normalzied locations names found in part 1. 

In [ ]:
# Standardize 'Time' column type to timedelta
weather_data['Time'] = pd.to_timedelta(weather_data['Time'])

# Normalize location names 
weather_data['Location'] = weather_data['Location'].replace({
    'Monte Carlo': 'Monaco',
    'Marina Bay': 'Singapore',
    'Miami Gardens': 'Miami',
    'Yas Island': 'Yas Marina'
})

**Merging Event & Weather**

- Using the **`StartTime`** from event data we can calulate the UTC time for each record in the weather dataset.

In [ ]:
# Add event name to handle repeat locations
weather_data = weather_data.merge(
    event_data[['Year', 'EventName', 'Location', 'StartTime']],
    on=['Year', 'EventName', 'Location'],
    how='left'
)

# Add a UTC lap start time for merging with weather data
weather_data['TimeUTC'] = weather_data['Time'] + weather_data['StartTime']

weather_data = weather_data.drop(['StartTime'], axis=1)

**Resampling Weather Data**

- Weather data is resampled for every second, interpolating the numeric weather values in-between each minute they are recorded.
- **`Rainfall`** is forward filled to retain the boolean value over the race.
- The resampled result is merged with our lap data set using the UTC column we previously created.

**Note**: One lap time occurred before any weather data was recorded and was filled using the nearest available timestamp.

In [ ]:
# Resampling function
def resample_func(weather):
    weather = weather.copy()
    weather.set_index('TimeUTC', inplace=True) # required for resample
    
    numeric_cols = ["AirTemp", "Humidity", "Pressure", "TrackTemp",
                    "WindDirection", "WindSpeed"]
    
    interpolate_df = weather[numeric_cols].resample("1s").interpolate()
    ffill_df = weather[["Rainfall"]].resample("1s").ffill()
    
    return interpolate_df.join(ffill_df)

In [ ]:
# Round TimeUTC down to the nearest second
weather_data["TimeUTC"] = weather_data["TimeUTC"].dt.floor("1s")

# Resample weather data
weather_resampled = (
    weather_data
    .groupby(['Year', 'Location', 'EventName'])
    .apply(resample_func, include_groups=False)
    .reset_index()
)

weather_resampled.head()

In [ ]:
# Round TimeUTC down to the nearest second
laps_raw["LapStartTimeUTC"] = pd.to_datetime(laps_raw["LapStartTimeUTC"]).dt.floor("1s")

# Missing weather data fill to closest available second
laps_raw.loc[
    laps_raw["LapStartTimeUTC"] == pd.to_datetime('2020-10-11 12:10:07'), 
    'LapStartTimeUTC'
] = pd.to_datetime('2020-10-11 12:10:27')

# Assign weather data per lap
lap_weather_data = pd.merge(
    left=laps_raw, 
    right=weather_resampled,
    left_on=['Year', 'Location', 'EventName', 'LapStartTimeUTC'],
    right_on=['Year', 'Location', 'EventName', 'TimeUTC'],
    how='left'
).drop('TimeUTC', axis=1)

lap_weather_data.head()

#### **Final Results**

- The combined dataset was stored as a pickle file to preserve data types and ensure consistency throughout the analysis.
- **File Location:** 'data/f1_lap_weather_data.pkl'

In [ ]:
# Store combined data
lap_weather_data.to_pickle('../data/f1_lap_weather_data.pkl')